# Ask 8 — Capstone: Your Pile, Answering for Real

Every stage from lessons 2–7, wired in order with the dials exposed, plus
the eval harness and the limits label. Swap in YOUR pile and ship it.

**Privacy first:** permission from whoever owns the documents; sensitive
parts removed before they go behind an answering system; and remember the
data path — every question plus its retrieved chunks goes to the model
provider at answer time. When in doubt, use a public pile.

In [ ]:
# The pile: documents from the (fictional) Jefferson High School.
# Real enough to search, small enough to read whole.
PILE = {
 "handbook_academics": """S4.1 Grading scale. A 90-100, B 80-89, C 70-79, D 60-69.
Semester grades weight exams at 30 percent.
S4.2 Exam Retake Policy. This policy applies to final exams only. Students
receive one retake per semester, requested within ten school days. The
higher score stands.
S4.3 Grade appeals. Appeals go to the department head in writing within
fifteen school days of the posted grade.
S4.5 Late work. Assignments lose 10 percent per school day late, to a
maximum of 50 percent. Teachers may grant extensions for documented
emergencies.""",
 "handbook_schedule": """S2.0 Bell schedule. Regular days run eight periods,
8:15 AM to 3:20 PM.
S2.1 Wednesday schedule. Dismissal at 1:30 PM every Wednesday for staff
development.
S2.4 Late arrival. Students arriving after 8:30 AM sign in at the main
office with a note.""",
 "handbook_trips": """S5.1 Field trips require a signed permission form
submitted five school days in advance.
S5.2 Trip costs above 20 dollars qualify for the student activity fund.
S5.4 Chaperones must be approved district volunteers.""",
 "handbook_athletics": """S6.2 Eligibility. Athletes must hold a C average
during their season. Freshmen may try out for varsity teams.
S6.3 Petitions. A varsity roster spot for a freshman requires a coach's
petition to the athletic director.""",
 "robotics_minutes": """Robotics club meets Tuesdays in room 214. Regional
trip is April 18; bring your signed permission form by April 10. Dues are
15 dollars for the year.""",
 "clubs_list": """Active clubs: robotics (Tuesdays), debate (Thursdays),
art collective (Fridays), chess (lunch, library). Sign-up forms at the
student office.""",
 "bus_routes": """Routes 12 and 15 serve the north side. Final pickup at
4:45 PM outside door C. Activity buses run Tuesday and Thursday only.""",
 "cafeteria": """Lunch periods run 11:10, 11:55, and 12:40. Breakfast is
served from 7:40 AM. Menus post monthly on the food services page.""",
}
print(f"{len(PILE)} documents, {sum(len(t) for t in PILE.values())} characters total")

## The dials — every choice you made this course, in one place

In [ ]:
CONFIG = {
    "overlap_sentences": 1,   # ask2: the seam insurance
    "top_k": 5,               # ask7: the k that captured the question set (3 left one answer at rank 4)
    "score_floor": 0.5,       # ask7: refuse below this - absence handled before the model
    "rewrite_queries": True,  # ask7: phrasing repair
}
print(CONFIG)

In [ ]:
def chunk_by_section(pile, overlap_sentences=1):
    """Cut on the S-section seams; carry a sentence of overlap across cuts."""
    chunks = []
    for doc, text in pile.items():
        parts, current, header = [], [], None
        for line in text.splitlines():
            if line.strip().startswith("S") and len(line) > 2 and line.strip()[1].isdigit():
                if current:
                    parts.append((header, " ".join(current)))
                header, current = line.strip().split()[0].rstrip("."), [line]
            else:
                current.append(line)
        if current:
            parts.append((header, " ".join(current)))
        for i, (header, body) in enumerate(parts):
            text_out = body
            if overlap_sentences and i > 0:
                prev_tail = parts[i-1][1].split(". ")[-1]
                text_out = prev_tail + " ... " + body
            chunks.append({"doc": doc, "section": header or doc, "text": " ".join(text_out.split())})
    return chunks

CHUNKS = chunk_by_section(PILE, overlap_sentences=CONFIG["overlap_sentences"])
print(f"{len(CHUNKS)} chunks")
for c in CHUNKS[:3]:
    print(f"  [{c['doc']} {c['section']}] {c['text'][:70]}...")

In [ ]:
import math, re, collections

def words(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 2]

# document frequency: in how many chunks does each word appear?
DF = collections.Counter()
for c in CHUNKS:
    for w in set(words(c["text"])):
        DF[w] += 1

def score(query, chunk):
    """Shared words, each weighted by rarity: rare words shout, common words whisper."""
    shared = set(words(query)) & set(words(chunk["text"]))
    return sum(1.0 / DF[w] for w in shared)

def retrieve(query, k=3):
    ranked = sorted(CHUNKS, key=lambda c: -score(query, c))
    return ranked[:k]

print("retriever ready")

In [ ]:
%pip install -q anthropic

In [ ]:
import os, getpass
# Ask your teacher for the class API key. getpass keeps it out of the file.
try:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Class API key: ")
    HAVE_KEY = len(os.environ["ANTHROPIC_API_KEY"]) > 10
except Exception:
    HAVE_KEY = False
print("Key loaded." if HAVE_KEY else "No key - precomputed outputs shown below each live cell.")

In [ ]:
MODEL = "claude-opus-5"

def llm(prompt, max_tokens=800):
    import anthropic
    client = anthropic.Anthropic()
    return client.messages.create(model=MODEL, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]).content[-1].text

## The assembled pipeline

In [ ]:
REWRITES = {"when do we get to leave early midweek": "what time is early dismissal on Wednesday"}

CONTRACT = """Answer using ONLY the sections provided below.
Cite the section id after each fact, like [S4.2].
If the sections do not contain the answer, reply exactly:
"That isn't in the provided documents." and say what the documents DO cover.

{sections}

Q: {question}"""

def ask_pile(question):
    q = REWRITES.get(question, question) if CONFIG["rewrite_queries"] else question
    # with the class key, rewriting unknown questions is one llm() call:
    # q = llm(f"Rewrite as a formal handbook query, reply with the query only: {question}")
    ranked = sorted(CHUNKS, key=lambda c: -score(q, c))
    best = score(q, ranked[0])
    if best < CONFIG["score_floor"]:
        return ("That isn't in the provided documents (no section scored above "
                f"the floor for this question)."), [], best
    used = ranked[:CONFIG["top_k"]]
    sections = "\n\n".join(f"[{c['section']}] ({c['doc']}) {c['text']}" for c in used)
    prompt = CONTRACT.format(sections=sections, question=question)
    if HAVE_KEY:
        return llm(prompt), used, best
    return f"(keyless: prompt built with {[c['section'] for c in used]})", used, best

for q in ["How many final exam retakes do I get?", "What does the ski trip cost?"]:
    answer, used, best = ask_pile(q)
    print("Q:", q)
    print("A:", answer, "\n")

## The eval harness — your question set becomes the grade

In [ ]:
QUESTIONS = [
    ("How many final exam retakes do I get?", "S4.2"),
    ("What is the late work penalty?", "S4.5"),
    ("When are field trip permission forms due?", "S5.1"),
    ("When does robotics club meet?", "robotics_minutes"),
    ("when do we get to leave early midweek", "S2.1"),
    ("What does the ski trip cost?", None),
]

rows = []
for q, want in QUESTIONS:
    answer, used, best = ask_pile(q)
    if want is None:
        grade = "refused-correctly" if "isn't in the provided" in answer else "SHOULD-HAVE-REFUSED"
    else:
        grade = "retrieved" if any(c["section"] == want or c["doc"] == want for c in used) else "retrieval-MISS"
    rows.append({"question": q, "grade": grade, "top_score": round(best, 2)})

hit_rate = sum(r["grade"] in ("retrieved", "refused-correctly") for r in rows) / len(rows)
for r in rows:
    print(f"  {r['grade']:18} (score {r['top_score']:4.2f})  {r['question']}")
print(f"\nmeasured rate: {hit_rate:.0%}")
assert hit_rate == 1.0, "on the Jefferson pile with these dials, all six should pass"

## The limits label — written from measurements, not hopes

In [ ]:
import pathlib
label = f"""# Limits label - Jefferson pile assistant
Covers: academics, schedule, trips, athletics, robotics, buses, cafeteria.
Does NOT cover: trip pricing, sports schedules, anything after 2026.
Measured on {len(QUESTIONS)} questions: {hit_rate:.0%} retrieved-or-refused-correctly.
Rules for users:
- A cited answer is checkable - follow the [section] before acting on it.
- An answer with no citation is a bug; report it.
- Questions and retrieved sections are sent to the model provider.
"""
pathlib.Path("limits_label.md").write_text(label)
print(label)

## Ship it

1. **The working notebook** — this one, with YOUR pile in the first cell
   and every dial defended by a number.
2. **The eval table** — your ten questions plus your user's new ones.
3. **The limits label** — regenerate after every dial change.
4. **The field report** — who used it, their three most surprising
   questions, what broke.

Leave it with its user. Check back in a week. Whether they're still using
it is the realest number this course produces.